In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/movie-recommendation-hackathon-2026/sample_submission.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/movies.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/imdb_data.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_tags.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_scores.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/train.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/test.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/tags.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/links.csv


In [2]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [3]:
train_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/train.csv')

train_df.head()

,userId,movieId,rating,timestamp
0,5163,57669,4.0,1518349992
1,106343,5,4.5,1206238739
2,146790,5459,5.0,1076215539
3,106362,32296,2.0,1423042565
4,9041,366,3.0,833375837


In [ ]:
tags_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/tags.csv')

tags_df.head()

,userId,movieId,tag,timestamp
0,3,260,classic,1439472355
1,3,260,sci-fi,1439472256
2,4,1732,dark comedy,1573943598
3,4,1732,great dialogue,1573943604
4,4,7569,so bad it's good,1573943455


In [5]:
movies_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/movies.csv')

movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
links_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/links.csv')

links_df.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [7]:
genome_tags_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_tags.csv')

genome_tags_df.head()

,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


In [8]:
genome_scores_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_scores.csv')

genome_scores_df.head()

,movieId,tagId,relevance
0,1,1,0.02875
1,1,2,0.02375
2,1,3,0.06250
3,1,4,0.07575
4,1,5,0.14075


## More EDA on my Part

In [9]:
# train_df['userId'].sort_values().unique().tolist()
grouped_data_users = train_df.groupby('userId')

grouped_data_users.get_group(106343)['rating'].sort_values(ascending=True).unique()

# You get all the movies a user has rated after grouping

array([0.5, 1. , 1.5, 2. , 2.5, 3. , 3.5, 4. , 4.5, 5. ])

In [10]:
tags_df['userId'].values

array([     3,      3,      4, ..., 162534, 162534, 162534])

In [11]:
grouped_data_tags = tags_df.groupby('userId')

grouped_data_tags.get_group(4)

# You get all the tags applied by a user to movies after grouping

,userId,movieId,tag,timestamp
2,4,1732,dark comedy,1573943598
3,4,1732,great dialogue,1573943604
4,4,7569,so bad it's good,1573943455
5,4,44665,unreliable narrators,1573943619
6,4,115569,tense,1573943077
7,4,115713,artificial intelligence,1573942979
8,4,115713,philosophical,1573943033
9,4,115713,tense,1573943042
10,4,148426,so bad it's good,1573942965
11,4,164909,cliche,1573943721


In [12]:
grouped_data_movies = movies_df.groupby('movieId')

grouped_data_movies.get_group(260)

,movieId,title,genres
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi


In [13]:
train_df['movieId'].sort_values().reset_index(drop=True).equals(movies_df['movieId'].sort_values().reset_index(drop=True))

False

In [14]:
movies_df[movies_df['movieId'] == 57669]

,movieId,title,genres
12111,57669,In Bruges (2008),Comedy|Crime|Drama|Thriller


## Function prototyping

In [ ]:
def cbf_recommender_fn(train_df, movie_df, user_id, rating_threshold):
    '''
    This function implements a recommender system for movies by
    analyzing what the movies the user likes and recommending
    movies similar to the one the user likes
    '''
    # Step one: Check if userId is valid
    if user_id not in train_df['userId']:
        return f"Please Enter a valid user Id"
    
    # Step two: Group the train_df by userId
    user_data = train_df.groupby('userId').get_group(user_id)

    # Step three: Use the rating_threshold to filter poorly rated movies
    filtered_user_data = user_data[user_data['rating'] >= rating_threshold]

    # Step Four: Reference movies in the movies_df after applying the filter
    movie_ids = filtered_user_data['movieId'].values

    if not pd.Series(movie_ids).isin(movie_df['movieId']).all():
        return "Please ensure the Movie Id is valid"
    
    movie_details = movie_df[movie_df['movieId'].isin(movie_ids)]

    # Step Five: Represent the movie_df genres column numerically
    genre_list = movie_details['genres'].str.split('|')

    mlb = MultiLabelBinarizer()
    genre_matrix = mlb.fit_transform(genre_list)

    # genre_df = pd.DataFrame(
    #     genre_matrix,
    #     columns=mlb.classes_,
    #     index=movie_details['title']
    # )

    cosine_sim = cosine_similarity(genre_matrix, genre_matrix)

    cosine_sim_df = pd.DataFrame(
        cosine_sim,
        index=movie_details['title'],
        columns=movie_details['title']
    )

    liked_positions = np.flatnonzero(
        movie_df['movieId'].isin(movie_ids).to_numpy()
    )

    return cosine_sim_df

In [ ]:
cbf_recommender_fn(train_df, movies_df, 106343, 1.0)